In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

Middleware is a mechanism that allows us to execute additional logic between different stages of an application's execution.


In simple words:

Middleware works like a security guard or checkpoint between two systems. It checks, modifies, monitors, or controls data before allowing it to move forward.

In [29]:
from langchain.chat_models import init_chat_model

from langchain.chat_models import init_chat_model

from langchain.agents import create_agent

llm= init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')

from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware

from langgraph.checkpoint.memory import MemorySaver

from langchain_core.messages import HumanMessage,SystemMessage



In [30]:

# msg based summerization
agent= create_agent(model=llm, 
checkpointer=MemorySaver(),
middleware=[SummarizationMiddleware(model=llm,trigger=('messages',10),keep=('messages',4))])

In [31]:
# run with threadId
config= {'configurable':{'thread_id':'test-1'}}

In [32]:
questions = [
    "What is LangGraph, and why do we use it in AI applications?",
    "What is the difference between LangChain and LangGraph?",
    "What is InMemorySaver in LangGraph?",
    "How do you import InMemorySaver in Python?",

]

for i, question in enumerate(questions, start=1):
    print(f"{i}. {question}")

1. What is LangGraph, and why do we use it in AI applications?
2. What is the difference between LangChain and LangGraph?
3. What is InMemorySaver in LangGraph?
4. How do you import InMemorySaver in Python?


In [33]:
for q in questions:
    responce=agent.invoke({'messages':HumanMessage(content=q)}, config=config)
    print(f'messages:{responce}', end=' ')
    print(f' length of messages:{len(responce['messages'])}', end= ' ')
    
    



messages:{'messages': [HumanMessage(content='What is LangGraph, and why do we use it in AI applications?', additional_kwargs={}, response_metadata={}, id='44cde3e6-bb39-43ef-aeaf-a5bb3dc68d87'), AIMessage(content='## What is LangGraph?\n\n**LangGraph** is an open‑source framework that turns *language‑model workflows* into **graphs**.  \nIt was built by the LangChain team (the same people behind the LangChain library) to give developers a declarative, composable, and highly‑observable way to design, run, and debug complex LLM‑powered applications.\n\n### Core Ideas\n\n| Concept | What it means | Why it matters |\n|---------|---------------|----------------|\n| **Nodes** | A unit of work – usually a prompt, a function call, or a call to an external API. | Keeps the logic modular and reusable. |\n| **Edges** | Directed connections that decide *what happens next* based on the current state. | Enables branching, loops, and conditional flows. |\n| **State** | A dictionary that carries everyt

In [ ]:

# token based summerization
agent= create_agent(model=llm, 
checkpointer=MemorySaver(),
middleware=[SummarizationMiddleware(model=llm,trigger=('tokens',250),keep=('messages',4))])

In [ ]:
for q in questions:
    responce=agent.invoke({'messages':HumanMessage(content=q)}, config=config)
    print(f'messages:{responce}', end=' ')
    print(f' length of messages:{len(responce['messages'])}', end= ' ')
    

messages:{'messages': [HumanMessage(content='What is LangGraph, and why do we use it in AI applications?', additional_kwargs={}, response_metadata={}, id='717f9eeb-f5a3-46d9-a949-59fc42ef609d'), AIMessage(content='## What is LangGraph?\n\n**LangGraph** is an open‑source framework built on top of the LangChain ecosystem that lets you design, run, and debug *stateful, graph‑based* AI applications.  \nIn a LangGraph program you:\n\n| Concept | What it means in LangGraph | Typical use‑case |\n|---------|----------------------------|-----------------|\n| **Node** | A unit of computation (e.g. “Ask LLM for a summary”, “Call a weather API”). | A step in a multi‑step workflow. |\n| **Edge** | A conditional or deterministic transition from one node to another. | “If the user says ‘I want the weather’, go to WeatherNode”. |\n| **State** | A dictionary that carries data (user query, LLM outputs, tool results, etc.) through the graph. | Keeps track of the conversation history, tool inputs, and int

In [35]:
# human middleware  human in loop

In [37]:
def read_email(email_id:str)->str:
    '''Mock function to read email by its ID'''
    return f'email content for ID {email_id}'

def send_email(recient:str,subject:str,body:str)->str:
    '''Mock function to send email by its ID'''
    return f'email send to {recient} with subject {subject}'


In [39]:

# token based summerization
agent= create_agent(model=llm, 
tools=[read_email, send_email],
checkpointer=MemorySaver(),
middleware=[HumanInTheLoopMiddleware(interrupt_on={
    'send_email':{'allowed_decisions':['approved','edit','reject']},
    'read_email':False
})
])

In [40]:
# run with threadId
config= {'configurable':{'thread_id':'test-approve'}}

In [43]:
agent.invoke(
    {'messages':[
HumanMessage(content='send email to mohitpayasi8085@gmail.com with subject  hello and body how are you')]
},config=config)

{'messages': [HumanMessage(content='send email to mohitpayasi8085@gmail.com with subject  hello and body how are you', additional_kwargs={}, response_metadata={}, id='ce5f3e1c-fcfd-44b9-b1b9-73f9276228b5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using function send_email. Provide body, recient, subject.', 'tool_calls': [{'id': 'fc_689761c4-9c30-4cd4-bc75-a3f265a642f3', 'function': {'arguments': '{"body":"how are you","recient":"mohitpayasi8085@gmail.com","subject":"hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 176, 'total_tokens': 237, 'completion_time': 0.064186527, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.008745804, 'prompt_tokens_details': None, 'queue_time': 0.155794515, 'total_time': 0.072932331}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': '